In [ ]:
# Shetti-AI: A High-Precision AI Framework for the Prediction of Functional Motif Mutation and Conservation
#
# Author: Sobhy H. (Manuscript in Preparation)
# Contact: [@hsobhy] / [hsobhy@live.com]
# https://orcid.org/0000-0002-8186-1085
#
#
# License: GNU Affero General Public License v3.0 (AGPL-3.0)
# This program is free software: you can redistribute it and/or modify
# it under the terms of the GNU Affero General Public License as
# published by the Free Software Foundation, either version 3 of the
# License, or any later version.
#
# This program is distributed WITHOUT ANY WARRANTY. 
# For non-commercial/commercial inquiries, contact the author.


In [2]:
!!pip install biopython

['Collecting biopython',
 '  Downloading biopython-1.87-cp313-cp313-win_amd64.whl.metadata (13 kB)',
 'Requirement already satisfied: numpy in .\\anaconda3\\Lib\\site-packages (from biopython) (2.2.0)',
 'Downloading biopython-1.87-cp313-cp313-win_amd64.whl (2.7 MB)',
 '   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--',
 '   --------------- ------------------------ 1.0/2.7 MB 9.5 MB/s eta 0:00:01',
 '   ---------------------------------------- 2.7/2.7 MB 9.5 MB/s  0:00:00',
 'Installing collected packages: biopython',
 'Successfully installed biopython-1.87']

In [10]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.SeqUtils import ProtParamData
import torch

# Your 15 input motifs
MOTIFS = [
    "LIVFMLIVFMLIVFM", "RHKSTALVSTPESRD", "CCWDDHHC",
    "NYEKCLIKNTAPYID", "LFLVIFVILLEYLLF", "ENTHNPSILVYADEG",
    "GKDESTNPLAAAAAL", "KRKKKKKKKKKRKKR", "TQAAAAAADAAIL",
    "YAAILAAAAAAYAAI", "AAAIQAAFRAAAAKK", "KRKKKKKKKKKRKK",
    "KKKKRRRRKKKKAAA", "KKSSDDDEEE", "LVQAAARGAAARK"
]

def get_pure_biopython_features(aa_char):
    kd_scale = ProtParamData.kd 
    if aa_char not in kd_scale:
        return [0.0] * 5
    
    analysis = ProteinAnalysis(aa_char)
    # 5 standard Biopython metrics
    return [
        analysis.molecular_weight() / 200.0,
        analysis.isoelectric_point() / 10.0,
        analysis.aromaticity(),
        analysis.instability_index() / 100.0,
        kd_scale[aa_char] 
    ]

# Create training tensor (15 motifs, 15 length, 5 features)
PROCESSED_DATA = torch.tensor([[get_pure_biopython_features(aa) for aa in m.ljust(15, 'A')[:15]] 
                               for m in MOTIFS], dtype=torch.float32)

In [ ]:
# modified part 1

In [14]:
# Use this updated extractor to keep numbers between -1 and 1
def get_pure_biopython_features(aa_char):
    kd_scale = ProtParamData.kd 
    if aa_char not in kd_scale: return [0.0] * 5
    
    analysis = ProteinAnalysis(aa_char)
    # We normalize these so the GAN doesn't "choke" on big numbers
    return [
        (analysis.molecular_weight() - 130) / 70, # Scale weight
        (analysis.isoelectric_point() - 7) / 5,    # Scale pI
        analysis.aromaticity() * 2 - 1,
        (analysis.instability_index() - 40) / 40,
        kd_scale[aa_char] / 4.5                    # Scale Hydrophobicity
    ]

In [11]:
import torch.nn as nn
import torch.nn.functional as F

class VAE_GAN(nn.Module):
    def __init__(self, seq_len=15, feat_dim=5, latent_dim=8):
        super(VAE_GAN, self).__init__()
        self.input_dim = seq_len * feat_dim
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(self.input_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, latent_dim * 2)
        )
        
        # Generator (Decoder)
        self.generator = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, self.input_dim),
            nn.Tanh()
        )
        
        # Discriminator
        self.discriminator = nn.Sequential(
            nn.Linear(self.input_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, x):
        h = self.encoder(x.view(-1, self.input_dim))
        mu, logvar = h.chunk(2, dim=-1)
        z = self.reparameterize(mu, logvar)
        gen_x = self.generator(z)
        return gen_x, mu, logvar

In [12]:
import torch.optim as optim

model = VAE_GAN()
opt_G = optim.Adam(list(model.encoder.parameters()) + list(model.generator.parameters()), lr=0.001)
opt_D = optim.Adam(model.discriminator.parameters(), lr=0.001)

real_data = PROCESSED_DATA.view(-1, 75)

for epoch in range(500):
    # Train Discriminator
    opt_D.zero_grad()
    gen_x, _, _ = model(real_data)
    d_loss = -torch.mean(model.discriminator(real_data)) + torch.mean(model.discriminator(gen_x.detach()))
    d_loss.backward()
    opt_D.step()
    
    # Train Generator
    opt_G.zero_grad()
    recon_x, mu, logvar = model(real_data)
    recon_loss = F.mse_loss(recon_x, real_data)
    kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    g_loss = recon_loss + 0.01 * kld_loss - torch.mean(model.discriminator(recon_x))
    g_loss.backward()
    opt_G.step()

In [ ]:
# WARNING: Severe Mode Collapse detected. 
# Discriminator overpowered Generator, causing the latent space to collapse 
# into low-variance biophysical signatures (repetitive A, S, G sequences).
# VAE reconstruction loss was insufficient to overcome the GAN gradient saturation.

In [13]:
def decode_to_string(generated_tensor):
    standard_aas = "ACDEFGHIKLMNPQRSTVWY"
    ref_lib = {aa: torch.tensor(get_pure_biopython_features(aa), dtype=torch.float32) for aa in standard_aas}
    
    results = []
    for i in range(generated_tensor.shape[0]):
        motif = ""
        for aa_feat in generated_tensor[i]:
            best_aa = min(ref_lib.keys(), key=lambda k: torch.dist(aa_feat, ref_lib[k]))
            motif += best_aa
        results.append(motif)
    return results

# Generate 15 unique mutations
model.eval()
with torch.no_grad():
    # We sample near the original motifs to get "Mutations" rather than random junk
    mu, logvar = model.encoder(real_data).chunk(2, dim=-1)
    mutated_latent = model.reparameterize(mu, logvar) 
    raw_gen = model.generator(mutated_latent).view(15, 15, 5)

final_outputs = decode_to_string(raw_gen)

print("="*40)
print(" GENAI MUTATED MOTIFS ")
print("="*40)
for i, m in enumerate(final_outputs):
    print(f"ORIGINAL: {MOTIFS[i]}")
    print(f"MUTATED : {m}\n")

 GENAI MUTATED MOTIFS 
ORIGINAL: LIVFMLIVFMLIVFM
MUTATED : AAAAAAAAAAAAAAA

ORIGINAL: RHKSTALVSTPESRD
MUTATED : SSSSSSSSSSSSSSG

ORIGINAL: CCWDDHHC
MUTATED : AAAAAAGAAAAAAAA

ORIGINAL: NYEKCLIKNTAPYID
MUTATED : SSGAGAAASGGGASS

ORIGINAL: LFLVIFVILLEYLLF
MUTATED : AAAAAAAAAAAAAAA

ORIGINAL: ENTHNPSILVYADEG
MUTATED : SSSSSSSSGSSSSSA

ORIGINAL: GKDESTNPLAAAAAL
MUTATED : SSSSSSSSGSGGGGA

ORIGINAL: KRKKKKKKKKKRKKR
MUTATED : SSSSSSSSSSSSSGG

ORIGINAL: TQAAAAAADAAIL
MUTATED : GGGGGGGAGAAGAAA

ORIGINAL: YAAILAAAAAAYAAI
MUTATED : GGGGGGGGGGGGGGG

ORIGINAL: AAAIQAAFRAAAAKK
MUTATED : GGGGGAGAGAGGAGG

ORIGINAL: KRKKKKKKKKKRKK
MUTATED : SSSSSSSSSSSSSSS

ORIGINAL: KKKKRRRRKKKKAAA
MUTATED : SSSSSSSSSSSSSSS

ORIGINAL: KKSSDDDEEE
MUTATED : GSSSSSSGAAAGAAA

ORIGINAL: LVQAAARGAAARK
MUTATED : GGGGGGGGGAAGAAA



In [ ]:
# ARCHITECTURE: VAE-Optimization with KLD-lite & MSE Reconstruction.
# STATUS: FAILED (Mode Collapse observed).
# ANALYSIS: Despite eliminating the Discriminator, the model collapsed into 
# neutral biophysical averages (A, S, G). This is attributed to a 'Latent 
# Bottleneck' (Dim=8 is too low) and MSE saturation, where the model 
# minimizes loss by predicting the statistical mean of the AA library.

In [18]:
# --- OPTIMIZED VAE-GAN SOLUTION ---
# PURPOSE: Solving Mode Collapse through Data Normalization, 
# LeakyReLU activations, and Deep Latent Space Training.

# 1. Normalized Feature Extraction
def get_scaled_features(aa_char):
    kd_scale = ProtParamData.kd 
    if aa_char not in kd_scale: return [0.0] * 5
    analysis = ProteinAnalysis(aa_char)
    # Scaling to -1 to 1 range ensures the AI "sees" the chemical differences
    return [
        (analysis.molecular_weight() - 130) / 70, 
        (analysis.isoelectric_point() - 7) / 5,
        analysis.aromaticity() * 2 - 1,
        (analysis.instability_index() - 40) / 40,
        kd_scale[aa_char] / 4.5
    ]

# 2. Data Preparation
processed_data = torch.tensor([[get_scaled_features(aa) for aa in m.ljust(15, 'A')[:15]] 
                               for m in MOTIFS], dtype=torch.float32)
real_flat = processed_data.view(-1, 75)

# 3. VAE-GAN Hybrid Model
class VAE_GAN_Final(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder: Motif -> Latent Space
        self.enc = nn.Sequential(nn.Linear(75, 128), nn.LeakyReLU(0.2), nn.Linear(128, 16))
        # Decoder/Generator: Latent Space -> New Mutation
        self.dec = nn.Sequential(nn.Linear(8, 128), nn.LeakyReLU(0.2), nn.Linear(128, 75), nn.Tanh())
    
    def encode(self, x):
        h = self.enc(x)
        return h[:, :8], h[:, 8:] # Split into Mean and Variance
    
    def decode(self, z):
        return self.dec(z).view(-1, 15, 5)

# 4. Deep Training Loop (2000 Epochs)
model_final = VAE_GAN_Final()
optimizer = optim.Adam(model_final.parameters(), lr=0.001)

for epoch in range(2001):
    optimizer.zero_grad()
    mu, logvar = model_final.encode(real_flat)
    # Reparameterization trick for generative sampling
    std = torch.exp(0.5 * logvar)
    z = mu + torch.randn_like(std) * std
    recon = model_final.decode(z)
    # Loss = Reconstruction Accuracy + Latent Regularization
    loss = nn.MSELoss()(recon.view(-1, 75), real_flat) + 0.05 * torch.sum(mu**2)
    loss.backward()
    optimizer.step()

# 5. Mapping Numbers back to Biopython Amino Acids
def final_decoder(gen_tensor):
    standard_aas = "ACDEFGHIKLMNPQRSTVWY"
    ref_lib = {aa: torch.tensor(get_scaled_features(aa)) for aa in standard_aas}
    results = []
    for motif_feat in gen_tensor:
        res = "".join([min(ref_lib.keys(), key=lambda k: torch.dist(f, ref_lib[k])) for f in motif_feat])
        results.append(res)
    return results

# 6. Generate and Predict
model_final.eval()
with torch.no_grad():
    mu, _ = model_final.encode(real_flat)
    # Adding 0.3 noise creates 'Mutations' instead of exact copies
    mutated_samples = model_final.decode(mu + torch.randn_like(mu) * 0.3)
    final_mutations = final_decoder(mutated_samples)

print("========================================")
print(" GENAI MUTATED MOTIFS (FIXED SOLUTION) ")
print("========================================")
for i, m in enumerate(final_mutations):
    print(f"ORIGINAL: {MOTIFS[i]}")
    print(f"MUTATED : {m}\n")

 GENAI MUTATED MOTIFS (FIXED SOLUTION) 
ORIGINAL: LIVFMLIVFMLIVFM
MUTATED : MCCCCCCCMCAMCCM

ORIGINAL: RHKSTALVSTPESRD
MUTATED : MMCCCCCCTCTYCCM

ORIGINAL: CCWDDHHC
MUTATED : MCCCCCCCMCTMCMM

ORIGINAL: NYEKCLIKNTAPYID
MUTATED : MMCCCCCCTCTMCCM

ORIGINAL: LFLVIFVILLEYLLF
MUTATED : MMCCCCCCMCTMCMM

ORIGINAL: ENTHNPSILVYADEG
MUTATED : MCCCCCCCMCCMCCM

ORIGINAL: GKDESTNPLAAAAAL
MUTATED : MMCCCCCCMCTFCMM

ORIGINAL: KRKKKKKKKKKRKKR
MUTATED : MMCCMCCCMCTMCMM

ORIGINAL: TQAAAAAADAAIL
MUTATED : MMCCCCCCMCTMCMM

ORIGINAL: YAAILAAAAAAYAAI
MUTATED : TYTCCCCCTCTYCMM

ORIGINAL: AAAIQAAFRAAAAKK
MUTATED : MCCFCCCCMCCMCFM

ORIGINAL: KRKKKKKKKKKRKK
MUTATED : TTTCMCCCTCTTCTM

ORIGINAL: KKKKRRRRKKKKAAA
MUTATED : MMCCCCCCMCTFCMM

ORIGINAL: KKSSDDDEEE
MUTATED : MCCCCCCCMCTMCMM

ORIGINAL: LVQAAARGAAARK
MUTATED : MMCCMCCCMCTMCMM



In [ ]:
# SWAP: Replaced Standard GAN with WGAN-Clipping.
# This prevents Mode Collapse by using Wasserstein Distance instead of binary 
# classification. Weight clipping (-0.01, 0.01) and 5:1 Critic-to-Generator 
# training ratio ensure stable gradients and diverse amino acid mutations.

In [16]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# 1. BIOPHYSICAL ENGINE
def get_biophysical_vector(seq):
    try:
        pa = ProteinAnalysis(seq)
        return np.array([
            pa.molecular_weight() / 1000.0, 
            pa.isoelectric_point() / 10.0,
            pa.aromaticity(),
            pa.instability_index() / 100.0,
            pa.gravy()
        ], dtype=np.float32)
    except: return np.zeros(5, dtype=np.float32)

AA_LIST = "ACDEFGHIKLMNPQRSTVWY"
AA_LIB = {a: get_biophysical_vector(a) for a in AA_LIST}

# 2. MODELS (WGAN STABILIZED)
class Generator(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 5)
        )
    def forward(self, z): return self.model(z)

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(5, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1)
        )
    def forward(self, x): return self.model(x)

# 3. TRAINING LOOP (FORCE DIVERSITY)
MOTIFS = [
    "LIVFMLIVFMLIVFM", "RHKSTALVSTPESRD", "CCWDDHHC", "NYEKCLIKNTAPYID", 
    "LFLVIFVILLEYLLF", "ENTHNPSILVYADEG", "GKDESTNPLAAAAAL", "KRKKKKKKKKKRKKR", 
    "TQAAAAAADAAIL", "YAAILAAAAAAYAAI", "AAAIQAAFRAAAAKK", "KRKKKKKKKKKRKK", 
    "KKKKRRRRKKKKAAA", "KKSSDDDEEE", "LVQAAARGAAARK"
]
real_data = torch.stack([torch.tensor(get_biophysical_vector(c)) for c in "".join(MOTIFS)])

latent_dim = 128
gen = Generator(latent_dim)
critic = Critic()

# Using RMSprop for WGAN stability
g_opt = optim.RMSprop(gen.parameters(), lr=0.00005)
c_opt = optim.RMSprop(critic.parameters(), lr=0.00005)

print("Training Shetti-AI...")
for epoch in range(4001):
    # (1) Update Critic more than Generator
    for _ in range(5):
        c_opt.zero_grad()
        real = real_data[torch.randint(0, len(real_data), (64,))]
        fake = gen(torch.randn(64, latent_dim)).detach()
        
        loss_c = -torch.mean(critic(real)) + torch.mean(critic(fake))
        loss_c.backward()
        c_opt.step()
        
        # Weight Clipping to prevent Critic explosion
        for p in critic.parameters():
            p.data.clamp_(-0.01, 0.01)

    # (2) Update Generator
    g_opt.zero_grad()
    gen_noise = torch.randn(64, latent_dim)
    loss_g = -torch.mean(critic(gen(gen_noise)))
    loss_g.backward()
    g_opt.step()

    if epoch % 1000 == 0:
        print(f"Epoch {epoch} | G_Loss: {loss_g.item():.4f} | C_Loss: {loss_c.item():.4f}")

# 4. MUTATION (With Position-Wise Randomization)
def mutate_motif(seq, temp=0.75):
    gen.eval()
    res = ""
    for char in seq:
        orig = get_biophysical_vector(char)
        # Unique noise per AA to prevent sequence-wide collapse
        z = torch.randn(1, latent_dim) 
        with torch.no_grad():
            gen_v = gen(z).numpy()[0]
        
        combined = (orig * (1 - temp)) + (gen_v * temp)
        res += min(AA_LIB.keys(), key=lambda a: np.linalg.norm(combined - AA_LIB[a]))
    return res

# 5. RESULTS
print("\n" + "="*50)
print(" Shetti-AI output")
print("="*50)
for m in MOTIFS:
    print(f"ORIGINAL: {m}")
    print(f"MUTATED : {mutate_motif(m)}")
    print("-" * 30)

Training Shetti-AI...
Epoch 0 | G_Loss: -0.0098 | C_Loss: -0.0001
Epoch 1000 | G_Loss: -0.0057 | C_Loss: -0.0038
Epoch 2000 | G_Loss: -0.0029 | C_Loss: -0.0024
Epoch 3000 | G_Loss: -0.0069 | C_Loss: -0.0013
Epoch 4000 | G_Loss: -0.0049 | C_Loss: -0.0017

 Shetti-AI output
ORIGINAL: LIVFMLIVFMLIVFM
MUTATED : AGHCLHGIKPSVSRC
------------------------------
ORIGINAL: RHKSTALVSTPESRD
MUTATED : AAKHRLSAHGHPHGA
------------------------------
ORIGINAL: CCWDDHHC
MUTATED : MPCDGPGI
------------------------------
ORIGINAL: NYEKCLIKNTAPYID
MUTATED : MGHKGVLKCCAGKWR
------------------------------
ORIGINAL: LFLVIFVILLEYLLF
MUTATED : LAAASPSSPGPMPLS
------------------------------
ORIGINAL: ENTHNPSILVYADEG
MUTATED : HRPPACHPSLGPRPL
------------------------------
ORIGINAL: GKDESTNPLAAAAAL
MUTATED : AMSPNPARPCPICPG
------------------------------
ORIGINAL: KRKKKKKKKKKRKKR
MUTATED : PCAHHLHMAATPRHH
------------------------------
ORIGINAL: TQAAAAAADAAIL
MUTATED : PASGLRGCDCKPR
-----------------------------